# TrackPulse — One-Time Streaming Silver Catch-Up to Historical Silver

**Run manually once, before enabling the recurring ADF trigger. Do not schedule this notebook every 12 hours.**

It uses an idempotent Delta `MERGE` on `event_id`: existing IDs are left untouched, and IDs missing from Historical Silver are inserted. It does not use `overwrite`. A schema mismatch fails before the write.

After the merge, it enables CDF on Streaming Silver so the recurring notebook can process future changes.

**Do not delete or reset any existing checkpoint.** The CDF job has its own independent checkpoint path.

In [0]:
# Purpose: Configure source and target Delta paths and validate the schemas before any write.
# Safety: Read-only preflight. This notebook must complete preflight before the one-time MERGE.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

streaming_silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

historical_silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry/"
)

if not DeltaTable.isDeltaTable(spark, streaming_silver_path):
    raise RuntimeError("Streaming Silver is not an existing Delta table. Stop before writing.")
if not DeltaTable.isDeltaTable(spark, historical_silver_path):
    raise RuntimeError("Historical Silver is not an existing Delta table. Stop before writing.")

streaming_silver_df = spark.read.format("delta").load(streaming_silver_path)
historical_silver_df = spark.read.format("delta").load(historical_silver_path)

source_types = {f.name: f.dataType.simpleString() for f in streaming_silver_df.schema.fields}
target_types = {f.name: f.dataType.simpleString() for f in historical_silver_df.schema.fields}
if source_types != target_types:
    raise ValueError(
        "Streaming Silver and Historical Silver schemas differ. No write was attempted. "
        f"Source schema: {source_types}; target schema: {target_types}"
    )
if "event_id" not in source_types:
    raise ValueError("Both tables must contain event_id as the stable merge key.")

null_or_blank_ids = streaming_silver_df.filter(
    F.col("event_id").isNull() | (F.trim(F.col("event_id")) == "")
).limit(1).count()
if null_or_blank_ids:
    raise ValueError("Streaming Silver contains null/blank event_id values. No write was attempted.")

source_rows = streaming_silver_df.count()
source_duplicate_ids = (
    streaming_silver_df.groupBy("event_id").count().filter(F.col("count") > 1).limit(1).count()
)
if source_duplicate_ids:
    raise ValueError("Streaming Silver contains duplicate event_id values. Resolve before merging Historical Silver.")

target_columns = historical_silver_df.columns
print("Streaming Silver rows available for one-time catch-up:", f"{source_rows:,}")
print("Streaming/Historical schemas match.")
print("Historical Silver path:", historical_silver_path)
print("No Historical Silver row count was run to avoid an unnecessary full scan.")


Streaming Silver rows available for one-time catch-up: 166,010
Streaming/Historical schemas match.
Historical Silver path: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/silver/telemetry/
No Historical Silver row count was run to avoid an unnecessary full scan.


In [0]:
# Purpose: Perform a one-time, idempotent catch-up from Streaming Silver into Historical Silver.
# Safety: Insert missing event IDs only; never overwrite or delete Historical Silver records.
# Note: The source is the smaller Streaming Silver table. This MERGE does not reload the 10M historical source data.

source_for_merge = streaming_silver_df.select(*target_columns)

pre_merge_history = (
    DeltaTable.forPath(spark, historical_silver_path)
    .history(1)
    .select("version")
    .first()["version"]
)

(
    DeltaTable.forPath(spark, historical_silver_path)
    .alias("t")
    .merge(source_for_merge.alias("s"), "t.event_id = s.event_id")
    .whenNotMatchedInsertAll()
    .execute()
)

latest_history_row = (
    DeltaTable.forPath(spark, historical_silver_path)
    .history(1)
    .select("version", "timestamp", "operation", "operationMetrics")
    .first()
)
print("One-time Historical Silver catch-up finished.")
print("Previous Historical Silver version:", pre_merge_history)
print("Latest Delta operation:", latest_history_row.asDict() if latest_history_row else None)
print("Existing event_id values were matched, not reinserted.")


One-time Historical Silver catch-up finished.
Previous Historical Silver version: 22
Latest Delta operation: {'version': 23, 'timestamp': datetime.datetime(2026, 10, 28, 20, 2, 8), 'operation': 'MERGE', 'operationMetrics': {'numTargetRowsCopied': '0', 'numTargetRowsDeleted': '0', 'numTargetBytesRemoved': '0', 'numTargetDeletionVectorsAdded': '0', 'numTargetRowsMatchedUpdated': '0', 'numTargetRowsMatchedDeleted': '0', 'numTargetRowsUpdated': '0', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'rewriteTimeMs': '2615', 'numTargetFilesAdded': '2', 'numTargetBytesAdded': '7616325', 'executionTimeMs': '2680', 'materializeSourceTimeMs': '15', 'numTargetRowsInserted': '166010', 'numTargetDeletionVectorsUpdated': '0', 'scanTimeMs': '0', 'numOutputRows': '166010', 'numTargetDeletionVectorsRemoved': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numSourceRows': '166010', 'numTargetFilesRemoved': '0'}}
Existing event_id values were matched, not reinserted.


In [0]:
# Purpose: Enable Change Data Feed on Streaming Silver for future incremental Historical integration.
# Safety: This changes a Delta table property only; it does not rewrite the existing rows.
# Run this one-time setup after the catch-up MERGE succeeds and before enabling the scheduled CDF job.

spark.sql(f"""
    ALTER TABLE delta.`{streaming_silver_path}`
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print("Change Data Feed has been enabled for future Streaming Silver changes.")
print("Source latest version:")
display(
    DeltaTable.forPath(spark, streaming_silver_path)
    .history(3)
    .select("version", "timestamp", "operation")
    .orderBy(F.desc("version"))
)


Change Data Feed has been enabled for future Streaming Silver changes.
Source latest version:


version,timestamp,operation
4,2026-10-28T20:02:11.000Z,SET TBLPROPERTIES
3,2026-10-28T19:38:36.000Z,MERGE
2,2026-10-28T18:08:01.000Z,MERGE
